# Lululemon Reddit: Data Scraping
**Authour:** Crystal Nguyen

**Date:** February 26, 2026

**Overview:** This notebook scrapes the 600 most recent posts from the r/lululemon subreddit using Reddit's public JSON API. For each post, the title, upvote score, body text, and URL are collected and saved to a CSV file to be used in the lululemon_sentiment_analysis notebook.

## Import Packages

In [1]:
# formally request content from website
import requests

# for data analysis and manipulation
import pandas as pd

# adds small delays between requests to avoid spamming servers
import time 

## Scrape Data

In [2]:
def scrape_lululemon_reddit(limit=600):
    """
    Scrapes the most recent posts from the r/lululemon subreddit using Reddit's public json API endpoint. 
    
    Sends paginated GET requests, collecting up to 100 posts per request until 600 limit is 
    reached. A 0.5 second delay is added between requests to avoid IP ban. For each post, the title, 
    upvote score, body text, and URL are extracted and stored. 
    
    Results are compiled into a dataframe and exported to a csv file for analysis.

    Parameters:
        limit (int): Maximum number of posts to scrape, default is 600

    Returns:
        DataFrame: Scraped posts with columns: title, score, text, url
    """
    # reddit's public json endpoint for the newest posts in r/lululemon
    url = "https://www.reddit.com/r/lululemon/new.json"

    # identify the request as a student project, required by Reddit's API rules
    headers = {"User-Agent": "BABI4005-student-project/1.0"}

    # empty list to store each post's data as we scrape
    results = []

    # tracks the last post seen so we can request the next page (pagination)
    after = None

    # keep scraping until we reach the post limit
    while len(results) < limit:

        # request up to 100 posts per page (Reddit's maximum)
        params = {"limit": 100}

        # if we have a previous page marker, add it to get the next page
        if after:
            params["after"] = after

        # send GET request to Reddit API and parse the json response
        response = requests.get(url, headers=headers, params=params)
        data = response.json()

        # extract the list of posts from the response
        posts = data["data"]["children"]

        # stop scraping if there are no more posts to retrieve
        if not posts:
            break

        # loop through each post and extract the fields we need
        for post in posts:
            p = post["data"]
            results.append({
                "title": p["title"],   # post headline
                "score": p["score"],   # upvote count
                "text": p["selftext"], # post body text (empty string if no body)
                "url": p["url"]        # link to the post or image
            })

            # wait 0.5 seconds between requests to avoid overloading Reddit's servers
            time.sleep(0.5)

        # get the unique ID of the last post to use as the starting point for the next page
        after = posts[-1]["data"]["name"]

    # convert results list to dataframe and trim to exact limit
    df = pd.DataFrame(results[:limit])

    # save to csv to use in sentiment analysis 
    df.to_csv("../data/raw_posts.csv", index=False)

    # confirm how many posts were saved
    print(f"Saved {len(df)} posts")

    # display the dataframe
    display(df)

    return df

## Run Scraper

In [3]:
df = scrape_lululemon_reddit()

Saved 600 posts


,title,score,text,url
0,new relaxed swiftly,3,relaxed long sleeved swiftly have been my go t...,https://www.reddit.com/r/lululemon/comments/1r...
1,Does this sweatshirt fit me?,10,Got gifted this oversized scuba half zip hoodi...,https://www.reddit.com/gallery/1rhlvf5
2,Lulu thrift store haul 28/2/26,26,"Scuba Oversized Full-Zip Hoodie (Army Green, X...",https://i.redd.it/87n7jhh7ecmg1.jpeg
3,Mahogany Muse In Person,9,I really appreciate when people post real pict...,https://www.reddit.com/gallery/1rhkmbx
4,Loose threads on a brand new pair of pants,0,The threads are coming loose on the seam of a ...,https://www.reddit.com/gallery/1rhki1j
...,...,...,...,...
595,Love month fit ❤️ Sequoia set + College Crimso...,46,Sequoia: Flow y (8)\n\nWT HR 25” (4)\n\nColleg...,https://www.reddit.com/gallery/1qvq5m6
596,Valentines Day Wrap Shirt for bigger girls?,5,Im loving the new Valentines Day Wrap Shirt (N...,https://www.reddit.com/r/lululemon/comments/1q...
597,Swiftly shirt,14,Are the swiftly tech 2.0 shirts see through? T...,https://i.redd.it/a2amq0yl2hhg1.jpeg
598,For all Black Plum lovers in the UK!,80,M&amp;S Heatgen Thermals in the colour Dark Gr...,https://i.redd.it/ev9fbsbkighg1.jpeg
